In [72]:
import geopandas as gpd
import pandas as pd
import geoplanar
import numpy as np
from shapely.ops import transform

source: https://data.public.lu/fr/datasets/r/d62942ec-0dc8-40e2-9fa4-27841e6e1ac1

In [73]:
gdf = gpd.read_file("/data/uscuni-ulce/extension2/lux/ACT2023v2_Buildings3d_Footprints.gpkg")

/home/eliska/extension2/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


In [61]:
gdf

,buildingid,gml_id,commune,lau2,modelmodificationdate,area,zmin,geometry
0,ACT_011f8a94-553f-4dd7-b1b3-91809e5e136f,ID_211f365a-3a47-476c-b3dd-9d6e6bd80c38,Luxembourg,0304,2024,482.32,298.42,"MULTIPOLYGON (((77212.617 75031.601, 77212.51 ..."
1,ACT_013e8081-397e-4d52-855e-776d6754496d,ID_15f9a498-3381-4e4c-84ce-9b00adf5ed38,Luxembourg,0304,2024,38.06,297.09,"MULTIPOLYGON (((76519.258 75066.678, 76515.071..."
2,ACT_018226b2-0845-45bb-b69e-12b2ffc6d4e7,ID_5ee52ae0-9e44-4104-b887-90e89c7efc0b,Luxembourg,0304,2024,384.37,299.03,"MULTIPOLYGON (((77431.485 75000.271, 77437.728..."
3,ACT_02e14eef-9fb8-4219-ae06-5b3b6a8abed4,ID_a30150bb-2937-46e2-95e1-b7f9514b59f4,Luxembourg,0304,2024,689.75,299.37,"MULTIPOLYGON (((76556.952 75155.362, 76557.2 7..."
4,ACT_0379d0df-261b-44c2-b860-a6e06e225bc4,ID_57ef2d99-065c-459c-a3d9-2daacdf1fc51,Luxembourg,0304,2024,159.90,297.65,"POLYGON ((76941.195 75059.456, 76939.221 75068..."
...,...,...,...,...,...,...,...,...
238184,ACT_b448fc4e-2b32-413d-a50e-3ad91b6c0166,ID_ed3f0d03-2b15-4724-b118-b56fed783834,Rosport-Mompach,1006,2024,475.19,303.43,"MULTIPOLYGON (((102302.261 90602.496, 102285.9..."
238185,ACT_ffddaea1-77f2-4bf9-95b3-f2247586635f,ID_e482f436-5403-4ecc-bf42-025a397e8610,Schuttrange,0307,2021,18.11,341.11,"MULTIPOLYGON (((85557.59 75253.197, 85554.787 ..."
238186,ACT_0a4228cd-9f61-47c9-b6c6-767ca46471b4,ID_4318a5f4-ba24-41f6-b9bd-f5bea06ac22a,Rosport-Mompach,1006,2021,1018.22,306.82,"MULTIPOLYGON (((102450.995 90313.324, 102443.8..."
238187,ACT_451d30a4-3c19-4996-8eb3-4a2c16732434,ID_5675ea45-c52c-4ab4-80ec-9baf6fb0d85e,Niederanven,0305,2021,198.15,386.65,"MULTIPOLYGON (((80468.92 79549.258, 80472.025 ..."


Check for weirdly large polygons

In [28]:
gdf["area"] = gdf.geometry.area
largest_100 = gdf.nlargest(100, "area")

In [ ]:
largest_100.explore()

Correct topology is the key, check for overlaps

In [74]:
gdf = gdf.explode(index_parts=False).reset_index(drop=True)

In [75]:
pairs = geoplanar.overlaps(gdf)
overlap_idx = np.unique(pairs)

In [76]:
gdf = geoplanar.trim_overlaps(gdf, strategy='smallest')

In [77]:
gdf = gdf[gdf.geometry.notna()]

# Drop sliver polygons below an area threshold
area_threshold = 0.5
gdf = gdf[gdf.geometry.area > area_threshold]

/tmp/ipykernel_198257/3828834630.py:1: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  gdf = gdf[gdf.geometry.notna()]


In [78]:
gdf_clean = gdf[["geometry"]].copy()

In [79]:
gdf_clean = gdf_clean.to_crs(epsg=3035)

In [80]:
gdf_clean

,geometry
0,"POLYGON ((4041367.376 2951494.242, 4041366.556..."
1,"POLYGON ((4040676.793 2951565.437, 4040672.55 ..."
2,"POLYGON ((4041584.328 2951451.539, 4041590.447..."
3,"POLYGON ((4040718.988 2951652.041, 4040719.098..."
4,"POLYGON ((4041097.766 2951536.218, 4041096.252..."
...,...
238184,"POLYGON ((4067221.933 2965737.408, 4067206.681..."
238185,"POLYGON ((4049712.071 2951280.484, 4049709.221..."
238186,"POLYGON ((4067355.599 2965440.871, 4067348.699..."
238187,"POLYGON ((4044851.143 2955836.258, 4044854.024..."


In [83]:
gdf_clean.to_parquet("/data/uscuni-ulce/extension2/lux/clean_3035.parquet")